# EBIM1 — Recuperación de información con NFCorpus

Este notebook implementa los requisitos de `RI_Ex.ipynb`: indexar documentos completos conservando sus IDs originales; recuperar con TF-IDF y coseno y con BM25; evaluar ambos modelos con el split `test` mediante Precision@10, Recall@10 y NDCG@10; y exponer `/health` y `/search` mediante FastAPI.

La evaluación utiliza únicamente consultas con qrels en `test`. Los resultados dependen del preprocesamiento y de los parámetros BM25 documentados abajo. La publicación en internet requiere desplegar el servicio en una plataforma accesible y mantenerlo activo.


## 1. Entorno y carga de datos

Abre Jupyter desde `C:\notebooks_IR` o deja que `ROOT` localice el corpus desde la ubicación del notebook. Archivos esperados: `nfcorpus/corpus.jsonl`, `queries.jsonl` y `qrels/test.tsv`.


In [7]:
from pathlib import Path
import json, re, math
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from nltk.stem import PorterStemmer

ROOT = Path.cwd()
if not (ROOT / "nfcorpus").exists():
    ROOT = Path(__file__).resolve().parent if "__file__" in globals() else ROOT
DATA_DIR = ROOT / "nfcorpus"
CORPUS_PATH = DATA_DIR / "corpus.jsonl"
QUERIES_PATH = DATA_DIR / "queries.jsonl"
QRELS_PATH = DATA_DIR / "qrels" / "test.tsv"
for path in (CORPUS_PATH, QUERIES_PATH, QRELS_PATH):
    if not path.exists(): raise FileNotFoundError(f"No se encuentra {path}. Ejecuta el notebook desde C:\\notebooks_IR.")

def read_jsonl(path):
    with path.open(encoding="utf-8") as f: return [json.loads(line) for line in f if line.strip()]

raw_docs = read_jsonl(CORPUS_PATH)
docs = [d for d in raw_docs if str(d.get("text", "")).strip()]
doc_ids = [str(d["_id"]) for d in docs]
doc_by_id = {str(d["_id"]): d for d in docs}
queries = {str(q["_id"]): str(q.get("text", "")) for q in read_jsonl(QUERIES_PATH)}
qrels_df = pd.read_csv(QRELS_PATH, sep="	", dtype={"query-id":str,"corpus-id":str})
print(f"Documentos indexables: {len(docs):,}; consultas: {len(queries):,}; juicios test: {len(qrels_df):,}")
display(pd.DataFrame(docs[:3])[['_id','title','text']])


Documentos indexables: 3,633; consultas: 3,237; juicios test: 12,334


,_id,title,text
0,MED-10,Statin Use and Breast Cancer Survival: A Natio...,"Recent studies have suggested that statins, an..."
1,MED-14,Statin use after diagnosis of breast cancer an...,BACKGROUND: Preclinical studies have shown tha...
2,MED-118,Alkylphenols in human milk and their relations...,The aims of this study were to determine the c...


## 2. Preprocesamiento e índice invertido

Se convierten a minúsculas y se tokenizan secuencias alfanuméricas inglesas; se eliminan las stopwords inglesas de scikit-learn y se aplica Porter stemming. La misma transformación se aplica a consultas y documentos. El índice invertido guarda frecuencia por documento y el ID original. No se fragmentan los documentos.


In [8]:
STOPWORDS = set(ENGLISH_STOP_WORDS)
STEMMER = PorterStemmer()
TOKEN_RE = re.compile(r"[a-z0-9]+(?:'[a-z0-9]+)?")
def tokenize(text):
    return [STEMMER.stem(t) for t in TOKEN_RE.findall(str(text).lower()) if t not in STOPWORDS]

doc_tokens = [tokenize(d.get("title", "") + " " + d.get("text", "")) for d in docs]
doc_lengths = np.asarray([len(t) for t in doc_tokens], dtype=np.float64)
postings = defaultdict(dict)
df = Counter()
for i, terms in enumerate(doc_tokens):
    tf = Counter(terms)
    for term, freq in tf.items(): postings[term][i] = freq
    df.update(tf.keys())
N = len(docs)
avgdl = float(doc_lengths.mean())
print(f"Vocabulario: {len(postings):,}; longitud media: {avgdl:.1f} tokens")


Vocabulario: 18,989; longitud media: 159.8 tokens


## 3. Modelos de recuperación

Se implementan dos métodos léxicos independientes sobre el mismo índice y preprocesamiento.


## 3.1 Modelo TF-IDF con similitud coseno

TF-IDF pondera los términos según su importancia en la colección y normaliza los vectores a longitud uno. El producto punto entre el vector de consulta y los documentos equivale a la similitud coseno.


In [9]:
def identity(tokens): return tokens
tfidf = TfidfVectorizer(tokenizer=identity, preprocessor=identity, token_pattern=None,
                        lowercase=False, norm="l2", dtype=np.float32)
X_tfidf = tfidf.fit_transform(doc_tokens)

def tfidf_scores(query):
    terms = tokenize(query)
    if not terms: return np.zeros(N, dtype=np.float64)
    return np.asarray((X_tfidf @ tfidf.transform([terms]).T).toarray()).ravel()


## 3.2 Modelo BM25

BM25 suma las contribuciones de los términos de consulta, ajustando por frecuencia en el documento, rareza del término y longitud del documento. Se usan `k1=1.2` y `b=0.75`.


In [10]:
def bm25_scores(query, k1=1.2, b=0.75):
    scores = np.zeros(N, dtype=np.float64)
    for term in set(tokenize(query)):
        posting = postings.get(term)
        if not posting: continue
        dft = len(posting)
        idf = math.log1p((N - dft + 0.5) / (dft + 0.5))
        idx = np.fromiter(posting.keys(), dtype=int)
        freq = np.fromiter(posting.values(), dtype=np.float64)
        norm = freq + k1 * (1 - b + b * doc_lengths[idx] / avgdl)
        scores[idx] += idf * freq * (k1 + 1) / norm
    return scores


## 3.3 Ranking unificado

Esta función selecciona uno de los dos modelos y ordena por puntuación descendente, con desempate determinista por ID original.


In [11]:
def rank(scores, top_k=10):
    order = np.lexsort((np.asarray(doc_ids), -np.asarray(scores)))
    return [(doc_ids[i], float(scores[i])) for i in order if scores[i] > 0][:int(top_k)]

def search(model, query, top_k=10):
    model = model.lower()
    if model == "tfidf": scores = tfidf_scores(query)
    elif model == "bm25": scores = bm25_scores(query)
    else: raise ValueError("model debe ser 'tfidf' o 'bm25'")
    return rank(scores, top_k)


## 4. Búsqueda de ejemplo

Prueba consultas de texto libre. La tabla muestra título/fragmento, ID original y puntuación.


In [12]:
def result_table(model, query, top_k=10):
    rows=[]
    for pos,(doc_id,score) in enumerate(search(model,query,top_k),1):
        d=doc_by_id[doc_id]; text=str(d.get("text","" )).replace("\n"," ")
        rows.append({"rank":pos,"document_id":doc_id,"score":score,
                     "title":d.get("title") or text[:180],"snippet":text[:300]})
    return pd.DataFrame(rows)

consulta_demo = "dietary fiber and cardiovascular health"
display(result_table("tfidf", consulta_demo, 10))
display(result_table("bm25", consulta_demo, 10))


,rank,document_id,score,title,snippet
0,1,MED-1234,0.584155,Cardiovascular benefits of dietary fiber.,The relationship between dietary fiber and ris...
1,2,MED-973,0.485761,What is a high fiber diet?,There is no recognized definition of what cons...
2,3,MED-2291,0.478184,Viscous versus nonviscous soluble fiber supple...,PURPOSE: This review focuses on the health ben...
3,4,MED-1233,0.426810,Dietary fiber intake and risk of first stroke:...,BACKGROUND AND PURPOSE: Fiber intake is associ...
4,5,MED-2293,0.408613,Dietary Fiber Future Directions: Integrating N...,The CODEX Alimentarius definition of dietary f...
5,6,MED-3858,0.390167,Dietary fiber intake and risk of breast cancer...,BACKGROUND: Observational and preclinical stud...
6,7,MED-4094,0.377260,Dietary fiber and breast cancer risk: a system...,BACKGROUND: Evidence from case-control studies...
7,8,MED-1038,0.360122,Dietary fiber and personality factors as deter...,"We examined effects of fiber on stool output, ..."
8,9,MED-1582,0.330003,A Prospective Study of Long-term Intake of Die...,Background & Aims Increased intake of dietary ...
9,10,MED-1231,0.318931,Lower lifetime dietary fiber intake is associa...,BACKGROUND: Fiber intake is associated with lo...


,rank,document_id,score,title,snippet
0,1,MED-1234,13.521952,Cardiovascular benefits of dietary fiber.,The relationship between dietary fiber and ris...
1,2,MED-1231,11.792464,Lower lifetime dietary fiber intake is associa...,BACKGROUND: Fiber intake is associated with lo...
2,3,MED-2765,11.555961,The value of current nutrition information.,To prevent or delay the occurrence of chronic ...
3,4,MED-2291,11.280480,Viscous versus nonviscous soluble fiber supple...,PURPOSE: This review focuses on the health ben...
4,5,MED-4299,10.824901,Evolution of the human diet: linking our ances...,The evolution of the human diet over the past ...
5,6,MED-2020,10.235066,Effect of wheat bran on serum lipids: influenc...,OBJECTIVE: Wheat fiber appears to protect from...
6,7,MED-2293,10.120474,Dietary Fiber Future Directions: Integrating N...,The CODEX Alimentarius definition of dietary f...
7,8,MED-2295,10.037900,Trends in dietary fiber intake in the United S...,BACKGROUND: Intake of dietary fiber has been r...
8,9,MED-1582,9.928945,A Prospective Study of Long-term Intake of Die...,Background & Aims Increased intake of dietary ...
9,10,MED-1493,9.870294,Flaxseed - a miraculous defense against some c...,"Presence of omega-3, omega-6 rich oil, alpha-l..."


## 5. Evaluación en qrels `test`

P@10 divide los documentos relevantes recuperados entre 10; R@10 divide entre el total de documentos relevantes de la consulta. NDCG@10 usa los grados de relevancia de los qrels con ganancia `2^rel - 1`. Se promedian métricas por consulta que tenga al menos un juicio relevante. El split de prueba se usa para el reporte final, no para ajustar parámetros.


In [13]:
qrels_df["score"] = pd.to_numeric(qrels_df["score"])
qrels = {str(q): dict(zip(g["corpus-id"].astype(str), g["score"].astype(float)))
         for q,g in qrels_df.groupby("query-id")}

def dcg(rels): return sum((2**r - 1) / math.log2(i + 2) for i,r in enumerate(rels))
def evaluate_one(qid, model, k=10):
    relevant = qrels[qid]
    ranked = search(model, queries.get(qid,""), k)
    rels = [relevant.get(doc_id,0.0) for doc_id,_ in ranked]
    hits = sum(r > 0 for r in rels)
    ideal = sorted(relevant.values(), reverse=True)[:k]
    return {"query_id":qid,"model":model,"P@10":hits/k,
            "R@10":hits/max(1,sum(r>0 for r in relevant.values())),
            "NDCG@10":dcg(rels)/dcg(ideal) if dcg(ideal)>0 else 0.0}

eval_qids = sorted(q for q in qrels if q in queries and any(v>0 for v in qrels[q].values()))
per_query = pd.DataFrame([evaluate_one(qid,m) for m in ("tfidf","bm25") for qid in eval_qids])
summary = per_query.groupby("model")[["P@10","R@10","NDCG@10"]].mean().sort_index()
print(f"Consultas evaluadas: {len(eval_qids)}")
display(summary)
display(per_query.pivot(index="query_id",columns="model",values=["P@10","R@10","NDCG@10"]).head(10))


Consultas evaluadas: 323


,P@10,R@10,NDCG@10
model,,,
bm25,0.236533,0.155966,0.328478
tfidf,0.226935,0.150785,0.311279


P@10            R@10            NDCG@10          
model      bm25 tfidf      bm25    tfidf      bm25     tfidf
query_id                                                    
PLAIN-1008  0.0   0.0  0.000000  0.00000  0.000000  0.000000
PLAIN-1018  0.6   0.6  0.100000  0.10000  0.521098  0.527843
PLAIN-102   0.1   0.0  0.041667  0.00000  0.148686  0.000000
PLAIN-1028  0.0   0.0  0.000000  0.00000  0.000000  0.000000
PLAIN-1039  0.2   0.2  1.000000  1.00000  1.000000  1.000000
PLAIN-1050  0.0   0.0  0.000000  0.00000  0.000000  0.000000
PLAIN-1066  0.0   0.0  0.000000  0.00000  0.000000  0.000000
PLAIN-1088  0.2   0.2  0.400000  0.40000  0.553146  0.553146
PLAIN-1098  0.0   0.0  0.000000  0.00000  0.000000  0.000000
PLAIN-1109  0.3   0.5  0.030612  0.05102  0.469000  0.623574

## Interpretación de la evaluación en qrels `test`

En las 323 consultas evaluadas, BM25 obtuvo mejores promedios que TF-IDF en las tres métricas: P@10 = 0.237 frente a 0.227, R@10 = 0.156 frente a 0.151 y NDCG@10 = 0.328 frente a 0.311. La diferencia es pequeña, pero sugiere que BM25 fue ligeramente más eficaz para recuperar y ordenar documentos relevantes en este split. Ambos promedios siguen siendo moderados, así que el rendimiento varía entre consultas; conviene revisar `per_query` para encontrar ejemplos donde un modelo destaque o ninguno recupere documentos relevantes. Estos valores describen este conjunto de prueba y no garantizan el mismo resultado para consultas nuevas.
